# 자연어 메타데이터 필터(Self-query pattern)

사용자의 자연어에서 의미 검색어, 허용된 메타데이터 조건, 결과 개수를 구조화해
Chroma 필터로 변환합니다. classic의 `SelfQueryRetriever`와 내부 translator에
의존하지 않고, 최신 구조화 출력과 allow-list 변환기를 사용합니다.


> **2026-09-21 업데이트**
>
> 이 노트북은 `langchain 1.4.2`, `langchain-core 1.6.3`,
> `langchain-openai 1.6.2`, `langchain-chroma 1.1.0` 기준으로 다시 작성했습니다.
> LangChain v1에서 예전 `langchain.retrievers` 구현은 `langchain-classic`으로
> 이동했고 `langchain-community`도 보관 상태이므로, 새 코드에서는 두 패키지와
> `langchain-teddynote`에 의존하지 않습니다. 대신 `langchain-core`의 Runnable,
> 공급자별 파트너 패키지, 명시적인 검색 함수를 조합합니다.
>
> 공식 참고: [LangChain v1 변경 사항](https://docs.langchain.com/oss/python/releases/langchain-v1),
> [v1 마이그레이션](https://docs.langchain.com/oss/python/migrate/langchain-v1),
> [OpenAI 임베딩](https://docs.langchain.com/oss/python/integrations/embeddings/openai),
> [Chroma 통합](https://docs.langchain.com/oss/python/integrations/vectorstores/chroma)


In [ ]:
# 최초 1회만 주석을 해제하세요.
# %pip install -qU "langchain==1.4.2" "langchain-core==1.6.3" \
#   "langchain-openai==1.6.2" "langchain-chroma==1.1.0" python-dotenv


In [ ]:
import getpass
import os
from typing import Literal
from uuid import uuid4

from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda
from langchain_openai import OpenAIEmbeddings
from pydantic import BaseModel, Field

load_dotenv()
if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OPENAI_API_KEY: ")

CHROMA_CONFIGURATION = {"hnsw": {"space": "cosine"}}


In [ ]:
products = [
    Document(
        page_content="수분 가득한 히알루론산 세럼으로 피부 깊숙이 수분을 공급합니다.",
        metadata={"year": 2024, "category": "스킨케어", "user_rating": 4.7},
    ),
    Document(
        page_content="24시간 지속되는 매트 파운데이션으로 모공을 자연스럽게 커버합니다.",
        metadata={"year": 2023, "category": "메이크업", "user_rating": 4.5},
    ),
    Document(
        page_content="식물성 성분의 저자극 클렌징 오일이 메이크업을 부드럽게 제거합니다.",
        metadata={"year": 2023, "category": "클렌징", "user_rating": 4.8},
    ),
    Document(
        page_content="비타민 C 브라이트닝 크림이 칙칙한 피부톤을 밝혀줍니다.",
        metadata={"year": 2023, "category": "스킨케어", "user_rating": 4.6},
    ),
    Document(
        page_content="선명한 발색과 촉촉한 사용감의 롱래스팅 립스틱입니다.",
        metadata={"year": 2024, "category": "메이크업", "user_rating": 4.4},
    ),
    Document(
        page_content="SPF50+/PA++++ 톤업 선크림이 자외선으로부터 피부를 보호합니다.",
        metadata={"year": 2024, "category": "선케어", "user_rating": 4.9},
    ),
]
vectorstore = Chroma.from_documents(
    products,
    OpenAIEmbeddings(model="text-embedding-3-small"),
    collection_name=f"self-query-{uuid4().hex}",
    collection_configuration=CHROMA_CONFIGURATION,
)


## 자연어를 안전한 검색 계획으로 변환


In [ ]:
Category = Literal["스킨케어", "메이크업", "클렌징", "선케어"]


class ProductSearchPlan(BaseModel):
    semantic_query: str = Field(
        description="메타데이터 조건을 제거한 제품 내용 검색어. 없으면 '화장품'"
    )
    category: Category | None = Field(default=None)
    year: int | None = Field(default=None, ge=2000, le=2100)
    min_rating: float | None = Field(default=None, ge=1, le=5)
    limit: int = Field(default=4, ge=1, le=10)


plan_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "사용자 요청을 화장품 검색 계획으로 변환하세요. 허용 필드는 category, "
            "year, min_rating, limit뿐입니다. 명시하지 않은 조건은 null로 두세요.",
        ),
        ("user", "{question}"),
    ]
)
model_name = os.getenv("OPENAI_CHAT_MODEL", "gpt-5.4-mini")
model = init_chat_model(f"openai:{model_name}", temperature=0)
plan_chain = plan_prompt | model.with_structured_output(ProductSearchPlan)

example_plan = plan_chain.invoke(
    {"question": "2023년 스킨케어 중 평점 4.5 이상인 제품 2개를 추천해 주세요"}
)
example_plan.model_dump()


## 검색 계획을 Chroma 필터로 변환

모델이 만든 문자열을 그대로 DB 필터로 실행하지 않습니다. 코드가 허용한 필드와
연산자만 `$and`, `$eq`, `$gte` 문법으로 변환합니다.


In [ ]:
def to_chroma_filter(plan: ProductSearchPlan) -> dict | None:
    conditions: list[dict] = []
    if plan.category is not None:
        conditions.append({"category": {"$eq": plan.category}})
    if plan.year is not None:
        conditions.append({"year": {"$eq": plan.year}})
    if plan.min_rating is not None:
        conditions.append({"user_rating": {"$gte": plan.min_rating}})

    if not conditions:
        return None
    if len(conditions) == 1:
        return conditions[0]
    return {"$and": conditions}


def self_query_search(question: str) -> dict:
    plan = plan_chain.invoke({"question": question})
    where = to_chroma_filter(plan)
    documents = vectorstore.similarity_search(
        plan.semantic_query or "화장품",
        k=plan.limit,
        filter=where,
    )
    return {
        "plan": plan.model_dump(),
        "chroma_filter": where,
        "documents": documents,
    }


self_query_retriever = RunnableLambda(self_query_search).with_config(
    {"run_name": "safe_self_query"}
)


In [ ]:
questions = [
    "평점이 4.8 이상인 제품을 추천해 주세요",
    "2023년에 출시된 상품을 추천해 주세요",
    "카테고리가 선케어인 상품을 추천해 주세요",
    "메이크업 상품 중 평점이 4.5 이상인 상품을 추천해 주세요",
    "2023년에 출시된 상품 1개를 추천해 주세요",
]

for question in questions:
    result = self_query_retriever.invoke(question)
    print(f"\n질문: {question}")
    print("계획:", result["plan"])
    print("필터:", result["chroma_filter"])
    for doc in result["documents"]:
        print("-", doc.page_content, doc.metadata)
